# Chapter 4: Hybrid Search

**Estimated time: 15 minutes**

You build a support chatbot. A user asks "what does error code E-7829 mean". Your vector database returns five chunks about error handling in general. Helpful. None of them contain the actual row for E-7829. The answer is sitting right there in your docs, but vector search missed it because it searched by meaning, not by the exact code.

This is the blind spot of semantic search. It understands concepts and fumbles exact identifiers. Product names, SKUs, error codes, version numbers. The fix is hybrid search. Run keyword search and vector search in parallel, merge the results with Reciprocal Rank Fusion, and get the best of both.

Run every cell top to bottom. When you reach the "Try it yourself" section, change the parameters and watch the lesson break.

## What you will see in three minutes

You are going to ask two questions of the same corpus, back to back, through three different retrieval pipelines.

Question one is **exact-match**: `What does error code E-7829 mean?`

- **Pure vector search** misses the authoritative row in `error_codes.md` entirely. The top five chunks are the table header, the footer, and adjacent rows. The LLM answer says "the context does not contain information about error code E-7829", which is exactly the bug we set out to fix.
- **Pure BM25 keyword search** finds the E-7829 row because it matches the exact token. The LLM answer correctly explains that the SSO token has expired and you need to re-authenticate.
- **Hybrid search (alpha 0.5)** runs both in parallel and fuses the results with RRF. The E-7829 row lands in the top three and the LLM answer is correct.

Question two is **semantic**: `I want my money back. What do I do?`

- **Pure vector** finds the refund policy and the blog post.
- **Pure BM25** finds the tax invoice FAQ and the pause-subscription FAQ, because they contain the common phrase "how do I". BM25 does not understand that "money back" means "refund".
- **Hybrid** pulls the refund policy back to the top because the vector path nailed it.

Same corpus. Same embeddings. Same model. Two queries that attack the system from opposite directions. Hybrid search handles both because it always has two shots at finding the right answer. The one knob you will tune is `alpha`, the mix between vector and keyword. Start at 0.5 and see what happens when you push it to the extremes.

## Install the dependencies

Run the next cell once. In Colab it installs the Python packages fresh, including `rank-bm25` which powers the keyword side of the lesson. Locally it assumes you already ran `pip install -r requirements.txt` in your virtual environment.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Cleaning up any prior chromadb install (silent)...")
    !pip uninstall -y -q chromadb chroma-hnswlib langchain-chroma 2>/dev/null

    print("Installing dependencies (takes about 60 seconds)...")
    !pip install -q \
        langchain==0.3.14 \
        langchain-openai==0.2.14 \
        langchain-community==0.3.14 \
        langchain-text-splitters==0.3.5 \
        faiss-cpu==1.13.2 \
        rank-bm25==0.2.2 \
        pypdf==5.1.0 \
        langsmith==0.2.6 \
        pandas==2.2.2 \
        matplotlib==3.9.4
    print("Done.")
else:
    print("Local environment detected. Skipping pip install.")
    print("Make sure you have run 'pip install -r requirements.txt' in your venv.")

In [ ]:
import os, sys

if IN_COLAB:
    # Always force a fresh clone so updates on GitHub are picked up. Without
    # this, a stale repo from a prior session would keep running the old utils.
    !rm -rf rag-for-pms
    !git clone -q https://github.com/DDRXV/rag-for-pms.git
    os.chdir("rag-for-pms")
else:
    # Local Jupyter: already inside the repo. Walk up to root if we are in chapters/.
    if os.path.basename(os.getcwd()) == "chapters":
        os.chdir("..")

# Python caches imported modules in sys.modules. Drop any cached utils.* so the
# next import reads the freshly cloned file, not a stale copy from an earlier
# session.
for cached in [m for m in sys.modules if m.startswith("utils")]:
    del sys.modules[cached]

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
from utils.shared import get_keys
get_keys()

## What you just did

`get_keys` pulled your OpenAI, LangSmith, and Cohere keys out of Colab secrets and turned on LangSmith tracing for this notebook. Every retrieval call and every LLM call you run in this chapter gets logged automatically to the visual waterfall at [smith.langchain.com](https://smith.langchain.com). Keep that tab open. You will want it at the end of the chapter.

## Look at the corpus

Same six documents from every chapter in this series. Pricing, product guide, billing FAQ, error codes, refund policy, and one outdated blog post. This chapter focuses on `error_codes.md` because it contains the one class of content where vector search falls apart: exact string identifiers that carry no semantic meaning on their own.

In [ ]:
from utils.shared import load_corpus

docs = load_corpus()
for d in docs[:6]:
    print(f"  {d.metadata.get('source', 'unknown'):25s}  {len(d.page_content):5d} chars")

## Peek at the target row

Before any pipeline runs, look at the one row of `error_codes.md` that the lesson is about. This is the row a complete answer to the E-7829 question needs.

In [ ]:
error_codes = next(d for d in docs if d.metadata["source"] == "error_codes.md")

# Pull the exact line for E-7829 so we can see the ground truth.
for line in error_codes.page_content.splitlines():
    if "E-7829" in line:
        print(line)

That is the authoritative answer. "SSO token expired, please re-authenticate." The string `E-7829` appears exactly once in the whole corpus, in that one row. No document anywhere else mentions the code.

Watch what happens when we ask for it.

## Build the vector index once

Same chunking strategy as Chapter 1. We will reuse this index for every pipeline in this chapter so the only thing changing across the experiments is the retrieval method, not the chunks.

In [ ]:
from utils.shared import build_index, make_chunks

# Chapter 1 showed that chunk_size=500 with chunk_overlap=50 is a good fit
# for this corpus. Hold chunking constant so the lesson is about retrieval,
# not chunk boundaries.
index = build_index(chunk_size=500, chunk_overlap=50)

# We will also need the raw chunks to build the BM25 index below. Rebuild
# them with the same parameters so both indexes see the exact same content.
chunks = make_chunks(chunk_size=500, chunk_overlap=50)

## Pipeline 1: pure vector search

Start with the naive approach. Embed the exact-match query, run it against FAISS, return the top five chunks.

The hypothesis is that vector search will miss the E-7829 row because the embedding of the query `What does error code E-7829 mean?` does not land near the embedding of a single table row that says `E-7829 | SSO token expired`. The query is natural language. The target is a dense technical string. Their embeddings sit in very different neighborhoods.

In [ ]:
from utils.shared import search, show_results

question = "What does error code E-7829 mean?"
vector_results = search(index, question, k=5)
show_results(vector_results, question=question)

## What vector search missed

Look at the source column. Every top result is from `error_codes.md`, which is technically the right document. But look at the preview column. None of them are the E-7829 row. Vector search found the header, the footer, and a handful of unrelated rows that happen to sit near the header in embedding space.

Now watch the LLM try to answer from that.

In [ ]:
from utils.shared import generate_answer

answer_vector = generate_answer(vector_results[:3], question)
print(answer_vector)

The LLM answer is either empty, vague, or says the context does not contain the answer. This is the worst possible failure mode for a production RAG system. The answer is sitting in your corpus. Your retrieval layer did not find it. The LLM has no choice but to say "I do not know".

Vector search is not broken in general. It is broken for this specific class of query. A user who types an identifier expects exact matching, not semantic similarity. You need a second retrieval path that actually does exact matching.

## Pipeline 2: BM25 keyword search

BM25 is the keyword search algorithm that powers Elasticsearch, Lucene, and every search engine you have ever used. Two ideas, one formula.

The first idea is **term frequency**. How many times does a word from the query appear in the document? More occurrences means a stronger match. The formula has a diminishing returns curve so a document that repeats the word twenty times does not completely dominate a document that mentions it three times.

The second idea is **inverse document frequency**. How rare is the word across the whole corpus? Common words like "the" and "error" appear everywhere so they get a low weight. Rare words like `E-7829` appear in one document so they get a high weight. A single match on a rare word beats a hundred matches on a common word.

Build the BM25 index over the same chunks the vector index already indexed.

In [ ]:
from utils.shared import build_bm25_index

bm25_index = build_bm25_index(chunks)

Now run the same exact-match question through BM25 only. BM25 scores are rewards. Higher is better. The gradient runs green at the top (good match) to red at the bottom (weak match), which matches the cosine similarity gradient in the vector search table above. Both tables point the same way now, so you can compare them side by side without re-reading the legend.

In [ ]:
from utils.shared import bm25_search, show_bm25_results

bm25_results = bm25_search(bm25_index, question, k=5)
show_bm25_results(bm25_results, question=question)

The E-7829 row is now in the top three. BM25 found it because the query contains the literal token `e-7829` and that token appears in exactly one chunk in the whole corpus. The IDF weight for `e-7829` is effectively "this word is unique, the match is guaranteed meaningful". BM25 does not know or care what SSO means. It only knows that the query and this one chunk share a rare token.

Feed the BM25 top three to the same LLM and see what comes out.

In [ ]:
answer_bm25 = generate_answer(bm25_results[:3], question)
print(answer_bm25)

The answer is now correct. "SSO token has expired, re-authenticate through your identity provider." That is the exact content of the E-7829 row.

BM25 won here because the query was literally a token lookup. But BM25 has the opposite blind spot. Watch what happens on a semantic question where the user's words do not appear in the target document at all.

In [ ]:
semantic_question = "I want my money back. What do I do?"

print("BM25 top 3 for the semantic question:")
for doc, score in bm25_search(bm25_index, semantic_question, k=3):
    print(f"  {round(float(score),3):>7}  {doc.metadata['source']}")

print()
print("Vector top 3 for the same question:")
for doc, score in search(index, semantic_question, k=3):
    print(f"  {round(float(score),3):>7}  {doc.metadata['source']}")

BM25 ranks the tax invoice FAQ and the pause-subscription FAQ ahead of anything refund-related, because those chunks contain the literal phrase "how do I" and the word "do". BM25 has no idea that "money back" means "refund". Vector search, in contrast, puts `refund_policy.pdf` at the top because the embedding for "I want my money back" lands near the embedding for "refund policy".

Each method is strong exactly where the other is weak. The question now is how to run both at the same time and merge the results.

## Pipeline 3: hybrid search

Hybrid search runs both approaches in parallel. The same query goes to the vector index and to the BM25 index at the same time. Each returns its own ranked list. The keyword path finds exact matches. The vector path finds semantic matches. Some documents show up in both lists. Some show up in only one.

Now you need a fusion algorithm to merge the two ranked lists into a single final ranking. The most common choice is Reciprocal Rank Fusion, or RRF. One line of math, no tuning required for the basic version.

## Reciprocal Rank Fusion by hand

Before running the helper, let us compute RRF on paper. Three documents. Doc A, Doc B, Doc C. Two ranked lists, one from keyword search and one from vector search.

The RRF formula for a single document in a single list is:

`score = 1 / (k + rank)`

Where `k` is a smoothing constant, usually 60, and `rank` is the position of the document in that list (1 for first, 2 for second, and so on). Add the scores across every list the document appears in. Higher total means better.

The worked example below is the exact scenario from the video.

In [ ]:
# Worked example straight from the video script.
# Keyword search ranks: A first, B second, C third.
# Vector search ranks:  C first, A second, B third.
# Use k = 60 as the RRF smoothing constant.

k = 60

# Each list is (rank_in_keyword, rank_in_vector).
docs = {
    "Doc A": (1, 2),
    "Doc B": (2, 3),
    "Doc C": (3, 1),
}

print(f"{'doc':<6} {'kw':>5} {'vec':>5} {'kw score':>10} {'vec score':>10} {'total':>10}")
for name, (kw_rank, vec_rank) in docs.items():
    kw_score = 1 / (k + kw_rank)
    vec_score = 1 / (k + vec_rank)
    total = kw_score + vec_score
    print(f"{name:<6} {kw_rank:>5} {vec_rank:>5} {kw_score:>10.4f} {vec_score:>10.4f} {total:>10.4f}")

Read the total column. Doc A comes out on top because it ranked well in both lists (first in keyword, second in vector). Doc C landed second because it was first in vector but third in keyword. Doc B is last because it was never in the top two of either list.

The intuition: RRF rewards documents that rank well across both methods. A document that is number one in one list but missing from the other will still score lower than a document that ranks in the top three of both. That is the whole point. You want consensus across retrieval methods, not a single top hit that only one method found.

Now run the real thing on the corpus.

## Run hybrid search on the exact-match question

`hybrid_search` runs the vector index and the BM25 index in parallel, pulls ten candidates from each, fuses the two lists with RRF, and returns the top `k` results sorted by fused score. The `alpha` parameter is the mix. Alpha 0.5 weights vector and keyword equally. Alpha 1.0 is pure vector. Alpha 0.0 is pure BM25.

In [ ]:
from utils.shared import hybrid_search, show_hybrid_results

hybrid_results = hybrid_search(
    index,
    bm25_index,
    question,
    k=5,
    alpha=0.5,   # Equal weight: 0.5 vector, 0.5 keyword. Try 0.0 or 1.0 in the Try-it-yourself section.
    rrf_k=60,    # Standard RRF smoothing constant. Try 10 or 200 to see how the ranking shifts.
)
show_hybrid_results(hybrid_results, question=question)

The E-7829 row is now in the top three. BM25 pushed it there because it ranked high in the keyword list, and RRF carried that ranking into the fused output even though vector search never found it. Notice the `rrf_score` column is in the 0.01 to 0.02 range. Those are the raw fused scores. The absolute number is not meaningful on its own, but the relative ordering is what matters.

Feed the hybrid top three to the LLM.

In [ ]:
answer_hybrid = generate_answer(hybrid_results[:3], question)
print(answer_hybrid)

The hybrid answer correctly identifies the SSO token expiration and tells the user to sign back in through their identity provider. Same LLM, same question, same corpus. The only thing that changed was the retrieval method.

## Two queries, two failure modes

The E-7829 question was the keyword win. Now run the semantic question through all three pipelines and watch the scoreboard flip. The question is `I want my money back. What do I do?`. The target is `refund_policy.pdf`, which never uses the phrase "money back". The LLM will need to understand that the two phrases mean the same thing.

In [ ]:
print("=" * 70)
print(f"Question: {semantic_question}")
print("=" * 70)

print("\nPure vector (alpha=1.0):")
for doc, _ in hybrid_search(index, bm25_index, semantic_question, k=3, alpha=1.0):
    print(f"  {doc.metadata['source']}")

print("\nPure BM25 (alpha=0.0):")
for doc, _ in hybrid_search(index, bm25_index, semantic_question, k=3, alpha=0.0):
    print(f"  {doc.metadata['source']}")

print("\nHybrid (alpha=0.5):")
for doc, _ in hybrid_search(index, bm25_index, semantic_question, k=3, alpha=0.5):
    print(f"  {doc.metadata['source']}")

The pure BM25 column puts billing FAQ on top because "how do I" is a keyword match. Pure vector puts refund policy on top because the embedding understands the intent. Hybrid keeps refund policy at the top because vector ranked it highly and BM25 did not have a strong enough alternative to pull it down.

That is the whole promise of hybrid search. Two different failure modes, one merged ranking, one path that works for both kinds of queries.

In [ ]:
# And the LLM answer under hybrid on the semantic question.
answer_hybrid_semantic = generate_answer(
    hybrid_search(index, bm25_index, semantic_question, k=3, alpha=0.5),
    semantic_question,
)
print(answer_hybrid_semantic)

## Tuning the alpha weight

You do not have to give both methods equal weight. Alpha is the mix.

- **Alpha 1.0** is pure vector. Good when users ask natural language questions and rarely quote identifiers. Bad when your corpus is full of SKUs and error codes.
- **Alpha 0.0** is pure BM25. Good when queries are mostly exact tokens. Bad when users paraphrase or ask in their own words.
- **Alpha 0.5** is the default. Start here unless you have a reason not to.

Most production teams land between 0.3 and 0.7 after they measure what their users actually type. The way to find your number is to run it against real queries, not to guess from first principles.

Watch the E-7829 answer change as alpha slides from 1.0 to 0.0.

In [ ]:
for alpha_value in [1.0, 0.7, 0.5, 0.3, 0.0]:
    results = hybrid_search(index, bm25_index, question, k=3, alpha=alpha_value)
    found_e7829 = any("E-7829" in doc.page_content for doc, _ in results)
    status = "found" if found_e7829 else "missing"
    print(f"alpha={alpha_value:.1f}  E-7829 row in top 3: {status}")

At alpha 1.0 the E-7829 row is missing because the vector path dominates the fusion and the vector path never saw it. As soon as you give the keyword path enough weight (alpha 0.5 or lower in this corpus) the row climbs into the top three. A production team running pure vector search on a corpus full of error codes is shipping the wrong answer to every exact-code query and does not even know it.

The second thing to notice is that the exact value of alpha is not a knife edge. Anywhere from 0.0 to 0.5 works on this corpus for this query. That is what makes hybrid search a safe default in production. You do not have to tune it perfectly to see the lesson land. You just have to give the keyword path a real seat at the table.

## Open your LangSmith trace

Open [smith.langchain.com](https://smith.langchain.com), click into the `rag-for-pms` project, and find the most recent traces from this notebook. You will see both retrieval calls (vector and BM25) and every LLM call in one visual waterfall. You can compare the pure-vector retrieval call to the hybrid retrieval call side by side and see exactly which chunks each one returned. That is the fastest way to build intuition for hybrid search in production.

## What you can do at work on Monday

Hybrid search is the single fastest RAG quality win on any corpus that contains identifiers. Here are the three questions to ask on your next RAG review.

1. Does the retrieval layer use keyword search, vector search, or both? If the answer is "just vector", ask what happens when a user types an error code, a SKU, or a product name. If the team cannot answer, you have found a reliability gap.
2. What is the fusion method? If the answer is "we take the top results from each and deduplicate", ask whether they tried RRF. Simple concatenation loses the rank information that RRF preserves.
3. What is the alpha weight and how was it picked? If the answer is "we guessed", ask whether anyone measured retrieval quality on real user queries at different alpha values. This is a one-day experiment that often ships a measurable quality bump.

## Try it yourself

Pick any of these. Change the value in the relevant cell, rerun, watch what happens.

1. Change `alpha=0.5` in the main `hybrid_search` call to `alpha=1.0`. That is pure vector. Rerun the E-7829 question and look at the top three. The E-7829 row should disappear from the results. The LLM answer should go back to "the context does not contain information". This is the exact failure mode hybrid search fixes.
2. Change `rrf_k=60` in the main `hybrid_search` call to `rrf_k=10`. A smaller `k` makes the top rank dominate much more strongly because `1 / (10 + 1)` is 0.091 while `1 / (10 + 5)` is 0.067, which is a much bigger relative gap than the gap at `k=60`. Rerun and see whether the ranking order changes.
3. Change the `question` variable to `"I cannot log in with my Okta account"` and rerun the vector, BM25, and hybrid cells. This is a semantic question that should route to the E-7829 row via meaning, not keyword. Watch which pipeline finds the row and which one misses it.

## Homework

Two exercises you can do on your own. Each takes about fifteen minutes.

1. **Run the full comparison for a second exact-match query.** The corpus contains several error codes: `E-1001`, `E-2301`, `E-3005`, `E-4012`, `E-5001`, `E-6002`, `E-7829`. Pick one you have not used. Run it through vector, BM25, and hybrid using the same cell pattern you already have. Fill in a simple table for yourself with four columns: query, vector found the row (yes or no), BM25 found the row (yes or no), hybrid found the row (yes or no). You are building intuition for which kinds of identifiers vector search happens to catch on its own and which kinds it misses every time.

2. **Apply this to your own product.** Pick a corpus from your own company: help docs, internal wiki, product specs, whatever. Think of three user queries that contain identifiers your users actually type: order IDs, SKUs, internal ticket codes, model numbers. Ask your engineering team what retrieval method the current system uses. If it is pure vector, draft a one-page proposal for adding BM25 plus RRF. Use the exact alpha numbers from this notebook as a starting point. This is the single biggest quality win most RAG teams have not shipped yet.

## What is next

Your retrieval layer now catches both exact-match queries and semantic queries. But even with hybrid search, your top five results still have ordering problems. The right document might be in position five when it should be in position one. A small reordering at the top of the list makes a visible difference in the LLM answer.

Chapter 5 is about reranking. Take the top ten hybrid results, feed them to a cross-encoder or a reranker API, and let it do a second pass that scores each candidate against the query with full attention to both sides. See you there.